In [1]:
import os

if os.name == 'nt' and 'HOME' not in os.environ:
    os.environ['HOME'] = os.path.expanduser("~")

from chop.models import get_model
model = get_model("chronos-2", pretrained=True)


c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
from chop import MaseGraph
import torch
import chop.passes as passes

batch_size = 1

print(model.config.chronos_config)
c_len = model.config.chronos_config["context_length"]
out_patch = model.config.chronos_config["output_patch_size"]

my_dummy_inputs = {
    "context": torch.randn((batch_size, c_len)),
    "group_ids": torch.zeros((batch_size,), dtype=torch.long),
    "future_covariates": torch.zeros((batch_size, out_patch), dtype=torch.float32),
    "num_output_patches": 1,
}

mg = MaseGraph(
    model,
    hf_input_names=[
        "context",
        "group_ids",
        "future_covariates",
        "num_output_patches",
    ],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": my_dummy_inputs})

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


{'context_length': 8192, 'input_patch_size': 16, 'input_patch_stride': 16, 'max_output_patches': 64, 'output_patch_size': 16, 'quantiles': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99], 'time_encoding_scale': 8192, 'use_arcsinh': True, 'use_reg_token': True}
tensor([[ 1.5825, -0.2589,  1.4837,  ...,  0.3164, -1.5247,  0.0971]])
tensor([[ 1.5825, -0.2589,  1.4837,  ...,  0.3164, -1.5247,  0.0971]])
tensor([[False, False, False,  ..., False, False, False]])
tensor([[True, True, True,  ..., True, True, True]])
tensor([[ 1.5825, -0.2589,  1.4837,  ...,  0.3164, -1.5247,  0.0971]])
tensor([[ 1.5825, -0.2589,  1.4837,  ...,  0.3164, -1.5247,  0.0971]])
tensor([[ 1.5920, -0.2493,  1.4933,  ...,  0.3259, -1.5152,  0.1066]])
tensor([[2.5346, 0.0622, 2.2298,  ..., 0.1062, 2.2958, 0.0114]])
tensor([[1.0130]])
tensor([[ 1.2393, -0.2452,  1.1857,  ...,  0.3184, -1.1978,  0.1057]])
tensor([[ 1.2393, -0.2452,  1.1857,  ...,  0.31

In [3]:
# Add chronos_config to mg.model
setattr(mg.model, "config", model.config)
setattr(mg.model, "chronos_config", model.chronos_config)

# The GraphModule returns a plain dict; wrap forward() to restore Chronos2Output type
from chop.models.chronos2.modeling_chronos2 import Chronos2Output

_orig_forward = mg.model.forward
def _patched_forward(*args, **kwargs):
    output = _orig_forward(*args, **kwargs)
    if isinstance(output, dict) and not isinstance(output, Chronos2Output):
        return Chronos2Output(**output)
    return output
mg.model.forward = _patched_forward


# Prediction Test

In [4]:
import pandas as pd
context_df = pd.read_csv("https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_hourly/train.csv")
print("Input dataframe shape:", context_df.shape)
display(context_df.head())

Input dataframe shape: (353500, 3)


,item_id,timestamp,target
0,H1,1750-01-01 00:00:00,605.0
1,H1,1750-01-01 01:00:00,586.0
2,H1,1750-01-01 02:00:00,586.0
3,H1,1750-01-01 03:00:00,559.0
4,H1,1750-01-01 04:00:00,511.0


In [5]:
from chop.passes.module import report_trainable_parameters_analysis_pass

_, _ = report_trainable_parameters_analysis_pass(mg.model)

+----------------------------------------------------+------------------------+
| Submodule                                          |   Trainable Parameters |
+====================================================+========================+
| input_patch_embedding                              |                2548224 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.hidden_layer                 |                 150528 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.act                          |                      0 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.output_layer                 |                2360064 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.dropout                      |                      0 |
+---------------------------------------

In [6]:
# Prepare pipeline for prediction
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline(model=mg.model)

pred_df = pipeline.predict_df(context_df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9])
print("Output dataframe shape:", pred_df.shape)
display(pred_df.head())


Output dataframe shape: (9936, 7)


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,H1,1750-01-30 04:00:00,target,629.201172,612.392395,629.201172,644.896240
1,H1,1750-01-30 05:00:00,target,563.743042,542.415894,563.743042,582.205811
2,H1,1750-01-30 06:00:00,target,518.652039,501.579590,518.652039,544.118652
3,H1,1750-01-30 07:00:00,target,494.096680,465.114197,494.096680,510.497070
4,H1,1750-01-30 08:00:00,target,467.443481,443.921204,467.443481,493.039856


In [7]:
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline(model=model)

pred_df = pipeline.predict_df(context_df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9])
print("Output dataframe shape:", pred_df.shape)
display(pred_df.head())

Output dataframe shape: (9936, 7)


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,H1,1750-01-30 04:00:00,target,623.331238,600.635437,623.331238,646.886047
1,H1,1750-01-30 05:00:00,target,562.325806,537.949463,562.325806,585.711121
2,H1,1750-01-30 06:00:00,target,519.875427,498.041443,519.875427,545.457153
3,H1,1750-01-30 07:00:00,target,483.038849,460.275024,483.038849,511.406921
4,H1,1750-01-30 08:00:00,target,455.434418,434.940796,455.434418,488.453644


# Perform Fev Bench

In [8]:
import fev
from chronos import Chronos2Pipeline

# Define the benchmark task
task = fev.Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="m4_hourly",
    horizon=24,
)

# Run evaluation with mg.model
pipeline = Chronos2Pipeline(model=mg.model)
predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)

print(f"Inference time: {inference_time_s:.2f}s")

# Compute evaluation metrics against the test split
summary = task.evaluation_summary(predictions_per_window, "test")
display(summary)


Inference time: 55.41s


{'model_name': 'test',
 'dataset_path': 'autogluon/chronos_datasets',
 'dataset_config': 'm4_hourly',
 'horizon': 24,
 'num_windows': 1,
 'initial_cutoff': -24,
 'window_step_size': 24,
 'min_context_length': 1,
 'max_context_length': None,
 'seasonality': 1,
 'eval_metric': 'MASE',
 'extra_metrics': [],
 'quantile_levels': [],
 'id_column': 'id',
 'timestamp_column': 'timestamp',
 'target': 'target',
 'generate_univariate_targets_from': None,
 'known_dynamic_columns': [],
 'past_dynamic_columns': [],
 'static_columns': [],
 'task_name': 'm4_hourly',
 'test_error': 5.747720064359884,
 'training_time_s': None,
 'inference_time_s': None,
 'num_forecasts': 414,
 'dataset_fingerprint': '19e36bb78b718d8d',
 'trained_on_this_dataset': False,
 'fev_version': '0.7.0',
 'MASE': 5.747720064359884}

In [9]:
fev.leaderboard(summaries=[summary], baseline_model="test")

c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\fev\analysis.py:553: RuntimeWarning: Mean of empty slice
  return np.nanmean(wins, axis=1)  # [n_models, ...]


,win_rate,skill_score,median_training_time_s,median_inference_time_s,training_corpus_overlap,num_failures
model_name,,,,,,
test,NaN,0.0,NaN,NaN,0.0,0


# FX Graph Operation Trace & Fusion Analysis

In [4]:
import torch.fx as fx
from tabulate import tabulate

# Walk every node in the traced FX graph and print op / target / shape metadata
rows = []
for node in mg.fx_graph.nodes:
    op = node.op
    if op == "call_function":
        target = node.target.__name__
    elif op == "call_method":
        target = node.target
    elif op == "call_module":
        target = f"{node.target} ({type(mg.model.get_submodule(node.target)).__name__})"
    else:
        target = str(node.target)

    # Pull common shape metadata if it exists
    meta = node.meta.get("mase", None)
    out_shape = ""
    if meta is not None:
        results = meta.parameters.get("common", {}).get("results", {})
        if results:
            shapes = [str(v.get("shape", "")) for v in results.values() if isinstance(v, dict)]
            out_shape = ", ".join(shapes)

    rows.append([node.name, op, target, out_shape])

print(tabulate(rows, headers=["Node", "Op", "Target", "Output shape"], tablefmt="simple"))


Node                                                         Op             Target                                                       Output shape
-----------------------------------------------------------  -------------  -----------------------------------------------------------  ----------------
context                                                      placeholder    context                                                      [1, 8192]
group_ids                                                    placeholder    group_ids                                                    [1]
future_covariates                                            placeholder    future_covariates                                            [1, 16]
num_output_patches                                           placeholder    num_output_patches                                           [1]
size                                                         call_method    size                                          

In [5]:
from collections import defaultdict

# -----------------------------------------------------------------------
# Fusion pattern detection
# Patterns: producer op -> consumer op that are commonly fused in
# custom CUDA kernels (e.g. FlashAttention, fused GeGLU, fused LayerNorm)
# -----------------------------------------------------------------------

FUSION_PATTERNS = {
    ("linear", "gelu"):        "Linear+GELU (fused GeGLU / MLP block)",
    ("linear", "relu"):        "Linear+ReLU",
    ("linear", "silu"):        "Linear+SiLU",
    ("linear", "sigmoid"):     "Linear+Sigmoid",
    ("layer_norm", "linear"):  "LayerNorm+Linear (pre-norm block)",
    ("add", "layer_norm"):     "Add+LayerNorm (residual+norm, FlashAttention-style)",
    ("matmul", "softmax"):     "MatMul+Softmax (attention score fusion)",
    ("softmax", "matmul"):     "Softmax+MatMul (attention output fusion)",
    ("mul", "add"):            "Mul+Add (scale-and-shift, e.g. FiLM / instance norm)",
    ("add", "tanh"):           "Add+Tanh",
    ("matmul", "add"):         "MatMul+Bias-Add",
    ("bmm",  "add"):           "BMM+Bias-Add",
    ("scaled_dot_product_attention", "linear"): "SDPA+Projection (full attention block)",
}

def node_key(node):
    """Return a short string key for a node suitable for pattern matching."""
    if node.op == "call_function":
        return node.target.__name__
    if node.op == "call_method":
        return node.target
    if node.op == "call_module":
        mod = mg.model.get_submodule(node.target)
        return type(mod).__name__.lower().replace("linear", "linear")
    return node.op

# Build a user->consumer adjacency for single-output nodes
fusion_hits = defaultdict(list)

nodes = list(mg.fx_graph.nodes)
for i, node in enumerate(nodes):
    k1 = node_key(node)
    # Check all direct users
    for user in node.users:
        k2 = node_key(user)
        pattern = (k1, k2)
        if pattern in FUSION_PATTERNS:
            fusion_hits[FUSION_PATTERNS[pattern]].append(
                f"  {node.name}  →  {user.name}"
            )

print("=" * 72)
print("FUSION OPPORTUNITIES DETECTED IN CHRONOS-2 FX GRAPH")
print("=" * 72)
if not fusion_hits:
    print("No known patterns found.")
else:
    for pattern_label, occurrences in sorted(fusion_hits.items()):
        print(f"\n[{pattern_label}]  ({len(occurrences)} occurrence(s))")
        for occ in occurrences[:6]:          # cap at 6 examples per pattern
            print(occ)
        if len(occurrences) > 6:
            print(f"  ... and {len(occurrences)-6} more")

print(f"\nTotal unique pattern types found: {len(fusion_hits)}")


FUSION OPPORTUNITIES DETECTED IN CHRONOS-2 FX GRAPH

[Linear+ReLU]  (15 occurrence(s))
  input_patch_embedding_hidden_layer  →  input_patch_embedding_act
  input_patch_embedding_hidden_layer_1  →  input_patch_embedding_act_1
  encoder_block_0_layer_2_mlp_wi  →  encoder_block_0_layer_2_mlp_act
  encoder_block_1_layer_2_mlp_wi  →  encoder_block_1_layer_2_mlp_act
  encoder_block_2_layer_2_mlp_wi  →  encoder_block_2_layer_2_mlp_act
  encoder_block_3_layer_2_mlp_wi  →  encoder_block_3_layer_2_mlp_act
  ... and 9 more

[Mul+Add (scale-and-shift, e.g. FiLM / instance norm)]  (49 occurrence(s))
  mul_6  →  add_3
  mul_7  →  add_3
  mul_8  →  add_4
  mul_9  →  add_4
  mul_16  →  add_11
  mul_17  →  add_11
  ... and 43 more

Total unique pattern types found: 2


# Chrome Trace Profiler

Profiles the Chronos-2 model with `torch.profiler` and exports a Chrome-compatible JSON trace.
Open `chrome://tracing` in Chrome (or Edge) and load the exported file to explore the timeline.


In [9]:
import torch

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Check where mg.model parameters actually live
devices = {p.device for p in mg.model.parameters()}
print(f"mg.model parameter devices: {devices}")

# Check dummy inputs
for k, v in my_dummy_inputs.items():
    if isinstance(v, torch.Tensor):
        print(f"  my_dummy_inputs['{k}'].device = {v.device}")


torch.cuda.is_available(): True
CUDA device: NVIDIA GeForce RTX 2060
mg.model parameter devices: {device(type='cuda', index=0)}
  my_dummy_inputs['context'].device = cpu
  my_dummy_inputs['group_ids'].device = cpu
  my_dummy_inputs['future_covariates'].device = cpu


In [12]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# 1. Move the already-loaded Chronos-2 model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Model on: {next(model.parameters()).device}")

# 2. Move dummy inputs to the same device
inputs = {
    k: v.to(device) if isinstance(v, torch.Tensor) else v
    for k, v in my_dummy_inputs.items()
}

# 3. Warm-up
with torch.no_grad():
    for _ in range(5):
        model(**inputs)
if device.type == "cuda":
    torch.cuda.synchronize()

# 4. Build activity list
activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

print(f"Profiling on device: {device}  |  Activities: {[a.name for a in activities]}")

# 5. Profiling Context
with profile(
    activities=activities,
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for _ in range(5):
        with record_function("model_inference"):
            with torch.no_grad():
                model(**inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        prof.step()

# 6. Export for chrome://tracing
prof.export_chrome_trace("profiler_trace.json")
print("Trace saved to profiler_trace.json")


Model on: cuda:0
Profiling on device: cuda  |  Activities: ['CPU', 'CUDA']
Trace saved to profiler_trace.json


In [13]:
import json
import re
from collections import defaultdict
from tabulate import tabulate

# ── 1. Load trace ────────────────────────────────────────────────────────────
with open("profiler_trace.json") as f:
    trace = json.load(f)

events = trace.get("traceEvents", [])

# ── 2. Extract CUDA kernel events, sorted by stream then start time ──────────
cuda_kernels = [
    e for e in events
    if e.get("ph") == "X"                    # complete events
    and e.get("cat") in ("kernel", "gpu_memcpy", "cuda_runtime")
    and "ts" in e and "dur" in e
]
cuda_kernels.sort(key=lambda e: (e.get("args", {}).get("stream", 0), e["ts"]))

# ── 3. Kernel name → fusion category heuristics ──────────────────────────────
FUSION_HINTS = [
    # pattern (regex)              label
    (r"gemm|sgemm|hgemm|cutlass",  "GEMM"),
    (r"softmax",                    "Softmax"),
    (r"layer_norm|layernorm",       "LayerNorm"),
    (r"gelu|relu|silu|sigmoid",     "Activation"),
    (r"elementwise|elt_",          "Elementwise"),
    (r"add|bias",                   "Add/Bias"),
    (r"scaled_dot",                 "ScaledDotProduct"),
    (r"dropout",                    "Dropout"),
    (r"copy|memcpy",                "MemCopy"),
    (r"embedding",                  "Embedding"),
]

KNOWN_FUSEABLE_PAIRS = {
    ("GEMM",            "Activation"):       "Linear + Activation  →  fused linear-act kernel",
    ("GEMM",            "Add/Bias"):         "Linear + Bias-Add    →  fused bias epilogue (cuBLAS)",
    ("GEMM",            "LayerNorm"):        "Linear + LayerNorm   →  fused MLP norm block",
    ("Add/Bias",        "LayerNorm"):        "Residual + LayerNorm →  FlashAttention-style fused norm",
    ("GEMM",            "Softmax"):          "MatMul + Softmax     →  fused attention score",
    ("Softmax",         "GEMM"):            "Softmax + MatMul     →  fused attention output",
    ("Activation",      "GEMM"):            "Activation + Linear  →  fused GeGLU gate",
    ("ScaledDotProduct","GEMM"):            "SDPA + Projection    →  full attention block",
    ("GEMM",            "Dropout"):         "Linear + Dropout     →  fused training kernel",
    ("Add/Bias",        "Activation"):      "Bias-Add + Activation →  fused activation epilogue",
}

def categorize(kernel_name: str) -> str:
    name_lower = kernel_name.lower()
    for pattern, label in FUSION_HINTS:
        if re.search(pattern, name_lower):
            return label
    return "Other"

# ── 4. Slide a window over each stream, detect fuseable consecutive pairs ────
gap_threshold_us = 5.0   # kernels further than this apart (µs) are not "consecutive"

fusion_hits   = defaultdict(list)   # label → [(k1_name, k2_name, gap_us)]
stream_stats  = defaultdict(int)

for stream, group in groupby_stream := __import__("itertools").groupby(
    cuda_kernels, key=lambda e: e.get("args", {}).get("stream", 0)
):
    prev = None
    for ev in group:
        stream_stats[stream] += 1
        name = ev.get("name", "")
        cat  = categorize(name)
        ts, dur = ev["ts"], ev["dur"]

        if prev is not None:
            prev_end = prev["ts"] + prev["dur"]
            gap_us   = ts - prev_end
            if 0 <= gap_us <= gap_threshold_us:          # back-to-back on same stream
                prev_cat = categorize(prev["name"])
                pair_key = (prev_cat, cat)
                label    = KNOWN_FUSEABLE_PAIRS.get(pair_key)
                if label:
                    fusion_hits[label].append((prev["name"], name, round(gap_us, 2)))

        prev = ev

# ── 5. Print summary ─────────────────────────────────────────────────────────
print("=" * 80)
print("CUDA KERNEL FUSION OPPORTUNITIES  (from profiler_trace.json)")
print(f"Gap threshold: ≤ {gap_threshold_us} µs between consecutive kernels on same stream")
print("=" * 80)

if not fusion_hits:
    print("\nNo back-to-back fuseable kernel pairs found within the gap threshold.")
    print("Try increasing gap_threshold_us or check that CUDA kernels were captured.\n")
else:
    rows = []
    for label, hits in sorted(fusion_hits.items(), key=lambda x: -len(x[1])):
        avg_gap = sum(h[2] for h in hits) / len(hits)
        example = f"{hits[0][0][:40]}  →  {hits[0][1][:40]}"
        rows.append([label, len(hits), f"{avg_gap:.2f} µs", example])
    print(tabulate(rows,
                   headers=["Fusion Opportunity", "Count", "Avg Gap", "Example pair"],
                   tablefmt="simple"))

    print(f"\nTotal fuseable pairs found: {sum(len(v) for v in fusion_hits.values())}")

# ── 6. Stream occupancy overview ─────────────────────────────────────────────
print("\n── CUDA Stream Kernel Counts ──")
for s, cnt in sorted(stream_stats.items()):
    print(f"  Stream {s}: {cnt} kernels")


SyntaxError: invalid syntax (3123238113.py, line 62)